<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c42-header.png" alt="Nextia Learning · Vision AI: See, Count and Inspect" width="100%">

# Solution: Classify products and pets from photos

**[Classify products and pets from photos](https://learning.nextia-ai.com/courses/vision/m09/classify-products-and-pets-from-photos/)** · Vision AI: See, Count and Inspect · Module 9, case study 2 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will** classify 37 breeds of cats and dogs from real open photos, and compare four families of methods against the number of labelled photos per breed: a colour histogram, CLIP zero-shot, a linear probe (alone and with the prompts) and fine-tuning. Then you set a line for automatic answers from the cost of a mistake, and run the whole method on new photos.

| | |
|---|---|
| **Time** | About 2.5 hours with the lesson page. The notebook runs in about 5 minutes on a laptop after the downloads. |
| **Needs** | An internet connection. The setup downloads the photos from the Oxford site (about 790 MB, once) and two models (CLIP, 605 MB, and ResNet-18, 47 MB). No account and no key. |
| **You should know** | [Transfer learning](https://learning.nextia-ai.com/courses/vision/m02/transfer-learning/), [CLIP zero-shot](https://learning.nextia-ai.com/courses/vision/m03/clip-zero-shot/) and [Linear probes and few-shot](https://learning.nextia-ai.com/courses/vision/m03/linear-probes-and-few-shot/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1, Pillow 13.0.0, scikit-learn 1.9.1, PyTorch 2.15.0, timm 1.0.31 and open_clip 4.2.1, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*). Colab, Kaggle and Windows were not tested. |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/vision/m09/classify-products-and-pets-from-photos/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C42/M09-L02-classify-products-and-pets-from-photos/classify-products-and-pets-from-photos-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses, with PyTorch (CPU), timm and open_clip. It needs an internet connection. The first time, it can take 2 minutes: PyTorch is large.

You should see one line with the versions: numpy 2.5.4, scikit-learn 1.9.1, torch 2.15.0, timm 1.0.31 and open_clip 4.2.1. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 pillow==13.0.0 scikit-learn==1.9.1 matplotlib==3.11.2 torch==2.15.0 torchvision==0.29.2 timm==1.0.31 open-clip-torch==4.2.1
import warnings; warnings.filterwarnings("ignore")
import numpy, sklearn, torch, timm, open_clip
print("numpy", numpy.__version__, "| scikit-learn", sklearn.__version__, "| torch", torch.__version__,
      "| timm", timm.__version__, "| open_clip", open_clip.__version__)

## Setup: get the photos

Run the next cell. It downloads the photo archive of the **Oxford-IIIT Pet Dataset** from its original site (about 790 MB), checks its checksum, and unpacks only the 2,183 photos of this case study into a folder `pets-case-study`. It also downloads the course's helper `compare.py`. On a fast connection it takes 2 to 5 minutes. If you run the cell again, it uses the files that are already there.

You should see `Ready: 2183 photos of 37 breeds`. If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete `pets-case-study/images.tar.gz`, then run the cell again.

**Source and licence.** "The Oxford-IIIT Pet Dataset" by Omkar M. Parkhi, Andrea Vedaldi, Andrew Zisserman and C. V. Jawahar, <https://www.robots.ox.ac.uk/~vgg/data/pets/>, licensed under [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). The copyright of each photo stays with its owner. Anything you share that is made from the photos (a figure, a table with photo names) is CC BY-SA 4.0 too. Read `dataset.md` next to this notebook.

In [ ]:
# Setup: download the Oxford-IIIT Pet photos from their original site (about 790 MB, once), check the archive,
# and unpack only the photos of this case study into pets-case-study/images/.
import hashlib, os, shutil, subprocess, tarfile, urllib.request
from pathlib import Path

IMAGES_URL = "https://thor.robots.ox.ac.uk/pets/images.tar.gz"
IMAGES_SHA256 = "67195c5e1c01f1ab5f9b6a5d22b8c27a580d896ece458917e61d459337fa318d"
COMPARE_URL = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C42/compare.py"
COMPARE_SHA256 = "65e58d2fe043105e3ff09626861a8f64e5548039df760ea10d6fc02c71f8fec3"
# The photos of this case study: for each breed, its species and the photo numbers of each part.
# (A list made from the dataset: CC BY-SA 4.0, as the dataset.)
SUBSET = {
    "Abyssinian": {"species":"cat","pool":[10,16,101,104,108,121,133,138,139,141,142,145,150,156,166,170,177,183,185,192],"val":[1,100,110,119,120,137,160,165,168,195],"test":[31,44,45,47,50,52,55,60,63,70,71,84,86,88,95,96,97,98,205,213,216,219,221,223,230],"new":[26,78,91,92]},
    "american_bulldog": {"species":"dog","pool":[100,103,115,118,119,121,122,126,138,147,149,167,168,172,178,181,183,186,194,199],"val":[101,110,111,116,136,139,158,161,169,188],"test":[2,22,28,29,38,47,48,53,58,68,70,71,74,85,87,89,90,92,93,95,96,98,205,208,224],"new":[41,56,59,72]},
    "american_pit_bull_terrier": {"species":"dog","pool":[15,17,104,106,108,114,115,123,128,129,133,138,139,142,146,153,155,168,170,180],"val":[14,113,130,136,137,141,147,157,159,162],"test":[6,8,21,24,26,30,31,36,37,40,43,50,59,60,63,65,71,75,76,79,81,87,98,195,202],"new":[20,48,191,197]},
    "basset_hound": {"species":"dog","pool":[11,16,18,100,114,121,126,133,134,135,143,153,154,158,159,170,173,176,186,187],"val":[101,108,115,123,139,140,155,171,184,189],"test":[6,9,25,27,28,31,48,50,51,54,62,63,65,66,75,76,78,82,88,90,92,95,193,198,199],"new":[2,19,45,194]},
    "beagle": {"species":"dog","pool":[16,100,101,110,111,117,118,120,121,123,137,152,155,158,160,164,173,176,183,192],"val":[10,113,119,136,148,175,181,185,186,188],"test":[1,7,23,25,30,33,39,48,49,51,52,54,57,67,71,75,77,82,84,95,99,196,199,201,204],"new":[6,26,44,60]},
    "Bengal": {"species":"cat","pool":[14,18,101,102,105,125,127,132,141,144,145,146,148,154,159,164,180,185,190,191],"val":[10,107,110,111,115,119,150,161,168,178],"test":[1,4,27,30,36,40,51,52,53,56,65,67,68,70,71,72,80,81,95,96,98,193,195,198,199],"new":[35,43,66,76]},
    "Birman": {"species":"cat","pool":[15,102,105,106,112,115,119,124,145,146,153,154,158,159,163,168,172,176,183,190],"val":[12,118,131,144,161,170,175,180,181,185],"test":[2,4,21,29,30,32,35,45,48,52,61,64,69,73,75,77,78,82,85,96,99,191,196,198,201],"new":[19,41,58,94]},
    "Bombay": {"species":"cat","pool":[1,10,16,19,105,126,127,136,144,150,155,156,161,163,168,171,174,185,191,193],"val":[100,104,107,109,115,122,135,153,166,184],"test":[2,4,7,9,26,27,29,33,34,38,40,48,53,54,56,60,64,75,83,87,90,96,202,205,215],"new":[23,25,43,208]},
    "boxer": {"species":"dog","pool":[11,102,104,108,111,117,126,129,131,142,143,148,151,156,160,168,172,178,180,183],"val":[12,134,144,152,159,161,162,174,187,188],"test":[3,6,20,24,32,34,39,45,54,59,61,63,65,70,73,75,78,81,85,86,88,89,94,96,200],"new":[4,67,95,97]},
    "British_Shorthair": {"species":"cat","pool":[15,18,103,108,113,114,125,141,156,158,160,166,177,182,183,185,190,195,196,207],"val":[105,107,109,118,137,159,181,198,199,203],"test":[22,25,28,31,32,38,41,43,45,47,54,58,72,74,94,95,96,99,230,263,267,269,270,272,273],"new":[2,84,88,92]},
    "chihuahua": {"species":"dog","pool":[13,14,17,110,114,121,129,138,139,141,153,159,160,161,171,176,180,181,183,184],"val":[112,120,127,135,136,140,145,158,168,170],"test":[2,9,20,27,28,33,39,40,46,52,53,56,59,63,64,68,70,77,82,85,90,94,99,196,200],"new":[24,50,60,78]},
    "Egyptian_Mau": {"species":"cat","pool":[12,18,100,101,108,109,110,112,114,116,127,131,132,138,160,162,180,188,196,197],"val":[105,106,117,128,130,133,154,155,157,173],"test":[2,4,6,20,21,25,26,27,30,31,35,47,48,53,56,60,68,73,76,78,90,92,95,198,209],"new":[7,59,86,199]},
    "english_cocker_spaniel": {"species":"dog","pool":[10,14,17,106,110,112,113,125,134,141,144,145,148,150,152,154,166,178,183,186],"val":[109,115,122,127,133,149,155,157,171,190],"test":[1,5,20,24,29,31,34,41,48,50,58,60,68,69,76,78,86,87,92,95,96,99,191,196,199],"new":[28,74,79,93]},
    "english_setter": {"species":"dog","pool":[16,18,100,116,117,120,128,132,135,136,147,149,151,160,163,166,169,172,177,182],"val":[11,105,107,134,153,157,161,162,180,181],"test":[5,20,28,29,33,39,40,52,54,57,61,63,65,67,68,69,73,74,77,80,85,92,94,96,97],"new":[64,84,86,194]},
    "german_shorthaired": {"species":"dog","pool":[10,13,16,104,106,112,117,119,127,129,134,138,139,146,153,156,159,161,166,173],"val":[101,109,111,114,133,143,145,150,177,183],"test":[2,7,8,22,25,28,29,34,42,43,46,47,54,55,62,65,70,74,75,81,83,88,94,97,200],"new":[31,39,66,96]},
    "great_pyrenees": {"species":"dog","pool":[10,11,15,16,17,18,100,104,106,110,114,120,133,138,144,150,158,161,186,188],"val":[101,118,121,122,125,129,160,168,170,180],"test":[2,3,21,22,29,31,37,38,41,48,49,55,56,58,67,68,77,78,80,83,96,98,99,198,199],"new":[19,82,90,95]},
    "havanese": {"species":"dog","pool":[105,108,110,114,115,124,130,132,133,135,142,144,147,154,155,159,164,178,188,189],"val":[10,101,119,123,157,162,163,165,168,176],"test":[3,4,23,27,32,41,47,66,67,69,70,74,75,77,80,82,88,91,93,95,191,192,193,197,198],"new":[37,43,62,89]},
    "japanese_chin": {"species":"dog","pool":[107,108,110,118,124,125,129,133,138,139,150,151,154,162,163,173,174,177,180,183],"val":[10,15,100,116,142,147,160,165,166,168],"test":[8,19,22,23,27,29,31,41,43,48,56,59,62,65,70,72,73,75,82,85,91,93,95,98,198],"new":[9,38,63,94]},
    "keeshond": {"species":"dog","pool":[15,100,103,118,126,127,138,139,140,142,145,149,153,166,167,171,178,179,183,185],"val":[11,13,18,106,120,122,136,181,186,189],"test":[3,7,9,19,23,27,29,41,43,48,50,53,56,60,66,75,80,81,83,84,87,96,98,192,195],"new":[4,31,47,191]},
    "leonberger": {"species":"dog","pool":[111,115,116,119,122,134,135,144,145,146,151,152,155,157,158,160,161,168,177,187],"val":[12,13,17,104,124,140,156,170,176,180],"test":[1,19,24,25,26,28,34,42,46,50,55,57,61,65,69,70,80,90,92,94,97,193,194,195,199],"new":[32,39,47,54]},
    "Maine_Coon": {"species":"cat","pool":[17,19,101,107,111,117,134,151,153,157,169,171,180,186,189,196,197,204,210,211],"val":[10,112,124,127,138,141,143,168,214,219],"test":[5,6,8,23,27,28,32,42,48,55,60,61,66,75,82,84,85,89,91,238,246,247,254,263,265],"new":[21,49,51,71]},
    "miniature_pinscher": {"species":"dog","pool":[11,17,18,101,106,111,126,129,132,134,137,141,146,147,148,159,166,175,185,186],"val":[12,100,108,128,136,164,172,177,183,184],"test":[1,9,20,21,37,41,47,48,49,51,56,59,63,65,66,68,70,71,81,82,87,88,92,193,198],"new":[45,52,67,89]},
    "newfoundland": {"species":"dog","pool":[11,18,104,105,107,120,121,122,128,133,134,139,146,160,162,167,170,171,181,184],"val":[10,14,102,132,136,140,141,142,179,188],"test":[1,2,5,6,9,21,22,25,34,41,42,45,49,57,60,66,71,72,79,83,85,86,87,92,196],"new":[28,55,95,99]},
    "Persian": {"species":"cat","pool":[1,16,17,18,100,101,123,132,144,152,169,173,174,182,184,190,197,200,206,213],"val":[13,15,19,103,111,122,153,156,161,170],"test":[3,6,7,21,27,31,34,49,52,64,69,71,75,81,83,87,90,91,94,95,233,263,268,269,270],"new":[38,61,62,259]},
    "pomeranian": {"species":"dog","pool":[13,17,101,118,126,129,139,143,154,165,170,173,174,177,178,184,185,186,189,190],"val":[18,103,121,128,135,159,161,163,171,182],"test":[1,4,7,21,26,38,47,52,54,55,57,60,63,66,68,71,76,78,82,85,91,93,98,191,200],"new":[35,51,59,99]},
    "pug": {"species":"dog","pool":[14,103,104,109,112,115,117,119,122,123,131,135,143,144,163,164,170,179,181,190],"val":[18,101,106,133,147,169,183,185,187,191],"test":[4,9,20,21,23,26,28,29,35,36,39,48,56,60,65,74,75,84,93,95,96,99,194,196,200],"new":[2,7,43,198]},
    "Ragdoll": {"species":"cat","pool":[10,12,107,112,114,115,129,131,133,148,158,172,181,182,183,187,188,189,195,199],"val":[100,103,121,139,141,154,155,164,180,191],"test":[5,6,9,20,25,43,45,56,58,63,68,75,76,77,84,93,95,204,205,210,224,250,252,255,265],"new":[57,69,245,261]},
    "Russian_Blue": {"species":"cat","pool":[14,18,103,104,107,122,127,128,131,136,145,146,157,159,162,166,169,170,174,177],"val":[114,120,125,133,143,158,164,172,199,200],"test":[24,30,32,36,43,49,51,55,59,60,63,64,67,70,78,80,84,85,90,94,95,201,209,212,236],"new":[28,29,77,99]},
    "saint_bernard": {"species":"dog","pool":[11,14,103,105,107,114,121,123,132,136,142,152,153,155,161,163,166,169,180,187],"val":[101,110,120,124,140,144,151,164,167,190],"test":[1,2,3,4,7,22,31,37,49,58,62,65,69,70,71,74,76,78,85,90,96,98,99,192,199],"new":[33,50,56,83]},
    "samoyed": {"species":"dog","pool":[10,100,107,108,111,112,118,129,137,139,141,148,149,161,164,170,179,180,183,184],"val":[13,101,122,145,160,168,171,174,175,188],"test":[4,5,8,21,22,25,30,39,40,45,50,55,56,58,61,64,65,66,79,80,83,86,95,98,198],"new":[1,78,81,88]},
    "scottish_terrier": {"species":"dog","pool":[13,16,101,106,111,113,114,115,119,120,127,131,134,141,148,159,166,171,180,185],"val":[17,102,110,125,128,142,149,152,163,165],"test":[20,30,31,33,37,39,44,50,56,58,60,66,69,79,82,85,86,87,89,91,95,97,98,195,197],"new":[46,54,70,83]},
    "shiba_inu": {"species":"dog","pool":[14,103,110,111,118,125,127,135,136,139,142,156,159,164,168,172,178,180,191,192],"val":[107,116,133,134,147,152,176,181,183,185],"test":[5,7,21,25,26,27,30,34,35,38,50,53,54,55,57,59,60,65,66,67,71,73,83,87,212],"new":[33,36,78,86]},
    "Siamese": {"species":"cat","pool":[14,18,102,112,135,145,150,157,158,159,161,163,169,175,180,181,183,191,193,200],"val":[19,106,108,110,111,128,131,141,179,194],"test":[6,25,28,30,32,37,41,47,57,66,72,74,80,83,85,87,93,98,210,218,224,237,240,249,254],"new":[60,70,95,238]},
    "Sphynx": {"species":"cat","pool":[1,17,102,104,108,116,126,130,139,143,157,159,168,182,192,194,196,198,202,203],"val":[103,122,145,156,158,166,169,170,185,186],"test":[4,5,8,23,28,39,40,48,55,57,65,73,95,96,99,213,222,228,229,230,238,241,242,243,250],"new":[2,24,42,249]},
    "staffordshire_bull_terrier": {"species":"dog","pool":[12,13,14,16,104,106,113,130,135,144,149,151,155,174,175,176,177,181,189,191],"val":[10,17,102,103,121,140,158,163,165,179],"test":[4,7,8,9,23,37,40,43,54,56,64,65,66,67,69,73,76,77,83,84,91,92,96,97,202],"new":[30,89,98,200]},
    "wheaten_terrier": {"species":"dog","pool":[11,14,18,101,109,123,129,130,131,153,154,158,163,164,174,175,177,182,186,188],"val":[17,133,134,136,146,148,149,151,165,179],"test":[4,5,6,8,41,42,47,48,50,51,55,61,62,63,71,73,75,79,81,84,86,88,92,96,195],"new":[3,21,27,93]},
    "yorkshire_terrier": {"species":"dog","pool":[17,102,103,112,121,122,123,132,136,138,140,148,149,156,165,174,180,181,183,187],"val":[10,13,127,133,135,139,150,160,163,175],"test":[3,4,19,26,29,31,35,40,41,42,43,45,46,49,59,69,71,72,75,79,82,88,90,91,96],"new":[21,34,52,74]},
}

HERE = Path.cwd()
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("pets-case-study").resolve()
(PROJECT / "images").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def get(url, path, expected):
    if path.exists() and sha256(path) == expected:
        return
    local = HERE / path.name  # a copy next to the notebook, if there is one
    if local.exists() and local.resolve() != path.resolve():
        shutil.copy(local, path)
    else:
        print("Downloading", url, "...")
        download(url, path)
    if sha256(path) != expected:
        raise SystemExit(f"{path.name} has the wrong checksum. Delete it and run this cell again.")

wanted = {f"images/{breed}_{n}.jpg" for breed, parts in SUBSET.items()
          for part in ("pool", "val", "test", "new") for n in parts[part]}
if not all((PROJECT / w).exists() for w in wanted):
    archive = PROJECT / "images.tar.gz"
    get(IMAGES_URL, archive, IMAGES_SHA256)
    with tarfile.open(archive, "r:gz") as tar:
        for member in tar:
            if member.name in wanted:
                tar.extract(member, PROJECT, filter="data")
get(COMPARE_URL, PROJECT / "compare.py", COMPARE_SHA256)
os.chdir(PROJECT)
print(f"Ready: {len(wanted)} photos of {len(SUBSET)} breeds in pets-case-study/images.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, Pillow's `Image`, matplotlib and the course helper as `cp` (the comparison table). It defines `show(files)`, which draws photos in a row, and the check helpers `expect` and `expect_hash`: they tell you if your answer is right without showing the answer. Run the cell. It prints nothing.

In [ ]:
# Helpers: images, the comparison table, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import compare as cp
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)

def show(files, titles=None, size=2.2):
    """Draw photos in a row (at most 8), cut square in the middle."""
    files = list(files)[:8]
    fig, axes = plt.subplots(1, len(files), figsize=(size * len(files), size + 0.3))
    for i, (ax, f) in enumerate(zip(np.atleast_1d(axes), files)):
        im = Image.open(f).convert("RGB"); s = min(im.size)
        im = im.crop(((im.width - s) // 2, (im.height - s) // 2, (im.width + s) // 2, (im.height + s) // 2))
        ax.imshow(im); ax.axis("off")
        if titles:
            ax.set_title(str(titles[i]), fontsize=8)
    plt.show()

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The data

Every photo shows one cat or dog of one of 37 breeds. The course team chose a fixed subset of the official lists: for each breed, a **pool** of 20 labelled photos to train on, 10 **validation** photos to choose settings, 25 **test** photos to compare the methods, and 4 **new** photos for the end-to-end run. Photos in which a person detector found a person were left out.

In [ ]:
BREEDS = [b.lower() for b in SUBSET]            # the 37 labels, in the dataset's order
SPECIES = {b.lower(): v["species"] for b, v in SUBSET.items()}
PARTS = ("pool", "val", "test", "new")
photos = pd.DataFrame([{"file": f"images/{b}_{n}.jpg", "breed": b.lower(), "species": v["species"], "part": part}
                       for part in PARTS for b, v in SUBSET.items() for n in v[part]])
files = {p: list(photos.loc[photos.part == p, "file"]) for p in PARTS}
y = {p: photos.loc[photos.part == p, "breed"].values for p in PARTS}
print(photos.groupby("part", sort=False).size().to_dict())
print(len(BREEDS), "breeds:", sum(s == "cat" for s in SPECIES.values()), "cats and",
      sum(s == "dog" for s in SPECIES.values()), "dogs")

> **Check.** You should see `{'pool': 740, 'val': 370, 'test': 925, 'new': 148}` and `37 breeds: 12 cats and 25 dogs`.

Four pairs of breeds that look alike (the course team looked at these photos: they show no person):

In [ ]:
LOOK_ALIKES = ["american_pit_bull_terrier_6", "staffordshire_bull_terrier_4", "Birman_82", "Ragdoll_6", "British_Shorthair_43", "Russian_Blue_24", "american_bulldog_85", "boxer_6"]
show([f"images/{n}.jpg" for n in LOOK_ALIKES], [n.rsplit("_", 1)[0] for n in LOOK_ALIKES])

## 2. The baseline: a colour histogram

The simplest method of the course: a colour histogram of each photo and its 5 nearest labelled photos (from [Rules and colour histograms](https://learning.nextia-ai.com/courses/vision/m02/rules-and-colour-histograms/)). It reads about 2,000 photos and takes about 10 seconds.

> **Predict.** On Larkfield's 8 categories it reached 0.263. What will it reach on 37 breeds?

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

def colour_histogram(path, bins=(8, 4, 4)):
    """Hue, saturation and value in 8 x 4 x 4 bins; the square root (the course's baseline, Module 2)."""
    hsv = np.asarray(Image.open(path).convert("RGB").resize((128, 128)).convert("HSV")).reshape(-1, 3)
    h, _ = np.histogramdd(hsv, bins=bins, range=((0, 256), (0, 256), (0, 256)))
    return np.sqrt(h.ravel() / h.sum())

def accuracy(y_true, y_pred):
    return cp.classification_scores(list(y_true), list(y_pred), BREEDS)["accuracy"]

H = {p: np.stack([colour_histogram(f) for f in files[p]]) for p in ("pool", "test")}
knn = KNeighborsClassifier(n_neighbors=5, n_jobs=1).fit(H["pool"], y["pool"])
print("colour histogram + 5-NN, 20 photos per breed: accuracy", accuracy(y["test"], knn.predict(H["test"])),
      "| chance:", round(1 / 37, 3))

> **Check.** About 0.044, against 0.027 by pure chance. Colour does not tell breeds apart: a black cat and a black dog have almost the same histogram.

## 3. CLIP zero-shot

CLIP compares each photo with a short text for each breed, with no labelled photo at all. The next cell computes the CLIP embedding of every photo once (about 30 seconds on a laptop, longer on Colab) and keeps it in `cache/`.

In [ ]:
import time, torch, open_clip, timm
from huggingface_hub import hf_hub_download
torch.set_num_threads(4)
# CLIP ViT-B/32 (OpenCLIP, laion2b_s34b_b79k, MIT), pinned: the same model as the course (605 MB, downloaded once).
f = hf_hub_download("laion/CLIP-ViT-B-32-laion2B-s34B-b79K", "open_clip_model.safetensors",
                    revision="1a25a446712ba5ee05982a381eed697ef9b435cf")
clip, _, preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained=f)
clip.eval(); tokenizer = open_clip.get_tokenizer("ViT-B-32")

def unit(X):
    return X / np.linalg.norm(X, axis=1, keepdims=True)

all_files = [f for p in PARTS for f in files[p]]
if Path("cache/clip.npy").exists():
    E = np.load("cache/clip.npy")
else:
    start, E = time.time(), []
    with torch.no_grad():
        for i in range(0, len(all_files), 32):
            batch = torch.stack([preprocess(Image.open(f).convert("RGB")) for f in all_files[i:i + 32]])
            E.append(clip.encode_image(batch).numpy())
    E = np.concatenate(E).astype(np.float32)
    Path("cache").mkdir(exist_ok=True); np.save("cache/clip.npy", E)
    print(f"{len(all_files)} photos in {time.time() - start:.0f} s")
E = unit(E)
X = {}
i = 0
for p in PARTS:
    X[p] = E[i:i + len(files[p])]; i += len(files[p])
print({p: X[p].shape for p in PARTS})

> **Check.** Four parts with 512 numbers per photo.

Now three prompt sets. A prompt set is the text around the breed's name.

In [ ]:
def text_embeddings(template):
    texts = [template.format(name=b.replace("_", " "), species=SPECIES[b]) for b in BREEDS]
    with torch.no_grad():
        return unit(clip.encode_text(tokenizer(texts)).numpy())

def zero_shot(template, part="test"):
    T = text_embeddings(template)
    return np.array(BREEDS)[(X[part] @ T.T).argmax(1)]

for template in ["{name}", "a photo of a {name}.", "a photo of a {name}, a type of pet."]:
    print(f"{template!r:42}", "test", accuracy(y["test"], zero_shot(template)),
          "| validation", accuracy(y["val"], zero_shot(template, "val")))

> **Check.** Test accuracy 0.851 with the name only, 0.861 with "a photo of a …", 0.893 with "…, a type of pet".

> **Your turn.** Many breed names do not say whether the animal is a cat or a dog. Write a prompt set `species_prompt` that says it: use `{name}` for the breed and `{species}` for `cat` or `dog`, in the style of the last prompt above. Then run the check.

In [ ]:
species_prompt = "a photo of a {name}, a type of {species}."
print("test", accuracy(y["test"], zero_shot(species_prompt)), "| validation", accuracy(y["val"], zero_shot(species_prompt, "val")))

In [ ]:
expect("zero-shot test accuracy with the species", None if species_prompt is ... else accuracy(y["test"], zero_shot(species_prompt)), 0.896, tolerance=0.004)

## 4. A linear probe, and a probe with the prompts

A **linear probe** is a logistic regression on the CLIP embeddings (from [Linear probes and few-shot](https://learning.nextia-ai.com/courses/vision/m03/linear-probes-and-few-shot/)). With 37 classes, the course's setting C = 2 is not the best: the next cell chooses C on the validation photos.

In [ ]:
from sklearn.linear_model import LogisticRegression

def draw(n, seed):
    """n photos of each breed from the labelled pool, chosen at random."""
    rng = np.random.default_rng(seed)
    return np.concatenate([rng.permutation(np.where(y["pool"] == b)[0])[:n] for b in BREEDS])

# The regularisation C, chosen on the validation photos with all 20 photos per breed.
for C in (0.5, 2, 10, 50, 200):
    m = LogisticRegression(C=C, max_iter=5000).fit(X["pool"], y["pool"])
    print("C", C, "validation", accuracy(y["val"], m.predict(X["val"])))

> **Check.** C = 10 is the best on validation.

The probe learns from labels; the prompts know the breed names. We can use both: mix the two probabilities, with a weight `alpha` for the prompts. The course team chose `alpha` and the number of fine-tuning epochs on the validation photos:

In [ ]:
EPOCHS = 5   # chosen on the validation photos (the course team's run: best of 15 epochs)
ALPHA = {1: 0.5, 2: 0.5, 5: 0.25, 10: 0.25, 20: 0.25}   # the weight of the prompts, chosen on the validation photos for each size

> **Predict.** With 1 labelled photo per breed, will the probe beat zero-shot (0.896)? And the probe with the prompts?

The next cell trains each size 10 times with different random photos (once for 20, because the pool has exactly 20) and prints the mean test accuracy. It takes about 30 seconds.

In [ ]:
def softmax(a):
    a = a - a.max(1, keepdims=True); e = np.exp(a)
    return e / e.sum(1, keepdims=True)

T = text_embeddings("a photo of a {name}, a type of {species}.")   # the best prompt set

def probe_and_prompts(rows, alpha, part):
    """Probabilities of the probe and of the prompts, mixed: (1 - alpha) x log p(probe) + alpha x log p(prompts)."""
    m = LogisticRegression(C=10, max_iter=5000).fit(X["pool"][rows], y["pool"][rows])
    assert list(m.classes_) == BREEDS  # the same order as the prompts
    log_probe = np.log(m.predict_proba(X[part]) + 1e-9)
    log_prompts = np.log(softmax(100 * X[part] @ T.T) + 1e-9)
    return softmax((1 - alpha) * log_probe + alpha * log_prompts)

curve = {}
for n in (1, 2, 5, 10, 20):
    draws = range(10 if n < 20 else 1)
    probe = [accuracy(y["test"], np.array(BREEDS)[probe_and_prompts(draw(n, s), 0.0, "test").argmax(1)]) for s in draws]
    both = [accuracy(y["test"], np.array(BREEDS)[probe_and_prompts(draw(n, s), ALPHA[n], "test").argmax(1)]) for s in draws]
    curve[n] = (round(float(np.mean(probe)), 3), round(float(np.mean(both)), 3))
    print(f"{n:2} per breed: probe {curve[n][0]}   probe + prompts {curve[n][1]}")

> **Check.** The probe alone: 0.504 with 1 photo per breed, 0.872 with 20, still below zero-shot. The probe with the prompts: 0.9 with 1 photo, 0.912 with 20. The prompts carry the breed names; the labels add what the prompts miss.

## 5. ResNet-18: frozen and fine-tuned

ResNet-18 was trained on ImageNet, which has 22 of these 25 dog breeds as classes, and only 3 of the 12 cats. The next cell runs every photo through ResNet-18 once (about 40 seconds), keeps the output of its third block for fine-tuning, and trains a probe on its embeddings.

In [ ]:
# ResNet-18 (timm resnet18.a1_in1k, Apache-2.0 weights trained on ImageNet-1k), pinned.
f = hf_hub_download("timm/resnet18.a1_in1k", "model.safetensors", revision="491b427b45c94c7fb0e78b5474cc919aff584bbf")
def resnet():
    return timm.create_model("resnet18.a1_in1k", pretrained=True, pretrained_cfg_overlay={"file": f}).eval()
from torchvision import transforms as TF
to_input = TF.Compose([TF.Resize(224), TF.CenterCrop(224), TF.ToTensor(),
                       TF.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
net = resnet()
if Path("cache/layer3.npy").exists():
    L3, R = np.load("cache/layer3.npy"), np.load("cache/resnet18.npy")
else:
    start, L3, R = time.time(), [], []
    with torch.no_grad():
        for i in range(0, len(all_files), 32):
            x = torch.stack([to_input(Image.open(f).convert("RGB")) for f in all_files[i:i + 32]])
            h = net.layer3(net.layer2(net.layer1(net.maxpool(net.act1(net.bn1(net.conv1(x)))))))
            R.append(net.global_pool(net.layer4(h)).numpy()); L3.append(h.numpy().astype(np.float16))
    L3, R = np.concatenate(L3), np.concatenate(R).astype(np.float32)
    np.save("cache/layer3.npy", L3); np.save("cache/resnet18.npy", R)
    print(f"{len(all_files)} photos in {time.time() - start:.0f} s")
XR, M3, i = {}, {}, 0
for p in PARTS:
    XR[p], M3[p] = unit(R[i:i + len(files[p])]), L3[i:i + len(files[p])]; i += len(files[p])
frozen = LogisticRegression(C=10, max_iter=5000).fit(XR["pool"], y["pool"])
print("ResNet-18 frozen + logistic regression, 20 per breed: accuracy", accuracy(y["test"], frozen.predict(XR["test"])))

> **Check.** About 0.875.

Fine-tuning trains the last block and a new last layer, as in [Transfer learning](https://learning.nextia-ai.com/courses/vision/m02/transfer-learning/). It takes about a minute on a laptop.

In [ ]:
# Fine-tune: train layer4 and a new last layer (the course's recipe: AdamW, learning rate 0.001, batch 32,
# no augmentation). The frozen layers give the same output every epoch, so they ran once (M3 above).
torch.manual_seed(0)
net = resnet(); net.reset_classifier(len(BREEDS))
head = torch.nn.Sequential(net.layer4, net.global_pool, net.fc)
opt = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=1e-4)
rows = draw(20, 0)
Xp = torch.tensor(M3["pool"][rows].astype(np.float32))
yp = torch.tensor([BREEDS.index(b) for b in y["pool"][rows]])
g = torch.Generator().manual_seed(0)
start = time.time()
for epoch in range(EPOCHS):
    head.train()
    perm = torch.randperm(len(Xp), generator=g)
    for i in range(0, len(Xp), 32):
        b = perm[i:i + 32]
        loss = torch.nn.functional.cross_entropy(head(Xp[b]), yp[b])
        opt.zero_grad(); loss.backward(); opt.step()
head.eval()
with torch.no_grad():
    Xt = torch.tensor(M3["test"].astype(np.float32))
    finetuned = np.array(BREEDS)[torch.cat([head(Xt[i:i + 128]) for i in range(0, len(Xt), 128)]).argmax(1).numpy()]
print(f"ResNet-18 fine-tuned, {EPOCHS} epochs, 20 per breed: accuracy {accuracy(y['test'], finetuned)} "
      f"({time.time() - start:.0f} s)")

> **Check.** About 0.846: lower than the frozen probe. With 20 photos per breed, fine-tuning learns the training photos too well.

## 6. The comparison table

Every method on the same 925 test photos, with the course's helper. The `task` column is `pets`.

In [ ]:
rows20 = draw(20, 0)
chosen_test = probe_and_prompts(rows20, ALPHA[20], "test")
predictions = {
    ("colour histogram + 5-NN (20 per breed)", 0, 740, "none (our code)"): knn.predict(H["test"]),
    ("CLIP ViT-B/32 zero-shot (a photo of a {}, a type of cat or dog)", 605, 0, "MIT (weights)"): zero_shot("a photo of a {name}, a type of {species}."),
    ("CLIP ViT-B/32 + logistic regression (20 per breed)", 605, 740, "MIT (weights)"): np.array(BREEDS)[probe_and_prompts(rows20, 0.0, "test").argmax(1)],
    ("CLIP ViT-B/32 + logistic regression + prompts (20 per breed)", 605, 740, "MIT (weights)"): np.array(BREEDS)[chosen_test.argmax(1)],
    ("ResNet-18 frozen + logistic regression (20 per breed)", 47, 740, "Apache-2.0 (weights)"): frozen.predict(XR["test"]),
    (f"ResNet-18 fine-tuned, layer4 + last layer, {EPOCHS} epochs (20 per breed)", 47, 740, "Apache-2.0 (weights)"): finetuned,
}
table = cp.Table()
for (method, mb, labelled, licence), pred in predictions.items():
    table.add(cp.classification_row(method, y["test"], pred, "pets test, 925 photos", None, mb, labelled, licence,
                                    task="pets"))
table.show("pets", columns=("method", "value", "worst", "worst_value", "model_mb", "labelled_images"))

## 7. The worst breeds

The chosen method is the probe with the prompts. Which breeds does it get wrong?

In [ ]:
chosen = predictions[("CLIP ViT-B/32 + logistic regression + prompts (20 per breed)", 605, 740, "MIT (weights)")]
scores = cp.classification_scores(list(y["test"]), list(chosen), BREEDS)
recall = pd.Series({b: s["recall"] for b, s in scores["per_class"].items()}).sort_values()
print("The five worst breeds (recall on 25 test photos each):")
print(recall.head(5).to_string())

> **Check.** The worst breed is american_pit_bull_terrier, with a recall of 0.4.

> **Your turn.** Find the pair of breeds that the chosen method confuses most often (in either direction). Count the mistakes with `Counter` and put the pair, as a sorted tuple of two breed names, in `worst_pair`.

In [ ]:
from collections import Counter
pairs = Counter(tuple(sorted((str(t), str(p)))) for t, p in zip(y["test"], chosen) if t != p)
worst_pair = pairs.most_common(1)[0][0]
print(worst_pair, pairs[worst_pair], "mistakes")

In [ ]:
expect_hash("worst_pair", worst_pair, '94d4639dd6a45621')

## 8. From probabilities to a decision

The shelter fills in the breed automatically when the top probability reaches a **line**. Below the line, a person checks. Three breeds always go to a person, whatever the probability (the lesson page explains why). A check costs 0.50 dollars, and a wrong breed on a listing 5 dollars (assumptions). The next cell tries every line on the validation photos.

In [ ]:
ALWAYS_CHECK = ["american_pit_bull_terrier", "staffordshire_bull_terrier", "american_bulldog"]
CHECK_COST, WRONG_COST = 0.50, 5.00          # assumptions: a person's check; a wrong breed on a listing

def decide(proba, line):
    """auto-fill the top breed if its probability reaches the line and it is not on the always-check list."""
    top = np.array(BREEDS)[proba.argmax(1)]
    check = (proba.max(1) < line) | np.isin(top, ALWAYS_CHECK)
    return top, check

def cost_per_1000(proba, y_true, line):
    top, check = decide(proba, line)
    wrong_auto = int(((top != y_true) & ~check).sum())
    return round(float((check.sum() * CHECK_COST + wrong_auto * WRONG_COST) / len(y_true) * 1000), 1)

chosen_val = probe_and_prompts(rows20, ALPHA[20], "val")
lines = np.round(np.arange(0, 1.0001, 0.01), 2)
val_costs = [cost_per_1000(chosen_val, y["val"], line) for line in lines]
LINE = float(max(l for l, c in zip(lines, val_costs) if c == min(val_costs)))  # the highest of the cheapest
plt.plot(lines, val_costs); plt.axvline(LINE, color="grey", linestyle="--")
plt.xlabel("line: the lowest probability that is filled in automatically"); plt.ylabel("dollars per 1,000 photos (validation)")
plt.show()
print("line", LINE, "| validation cost per 1,000 photos:", min(val_costs))

> **Check.** The cheapest line on validation is 0.47, at 175.7 dollars per 1,000 photos.

> **Your turn.** Use `cost_per_1000` with the chosen method's test probabilities (`chosen_test`) and the line `LINE`. Put the cost in `test_cost`. Is it close to the validation cost?

In [ ]:
test_cost = cost_per_1000(chosen_test, y["test"], LINE)
print(test_cost)

In [ ]:
expect("test_cost", test_cost, 204.9, tolerance=0.02)

## 9. End to end on new photos

The function `suggest` is the whole method for one new photo: read it, embed it with CLIP, mix the probe and the prompts, and decide. The 148 new photos were never used to train or to choose anything. First, six of them:

In [ ]:
probe20 = LogisticRegression(C=10, max_iter=5000).fit(X["pool"][rows20], y["pool"][rows20])

def suggest(file):
    """One new photo -> the three most likely breeds and the zone: the whole method, end to end."""
    with torch.no_grad():
        e = unit(clip.encode_image(preprocess(Image.open(file).convert("RGB"))[None]).numpy())
    log_probe = np.log(probe20.predict_proba(e) + 1e-9)
    log_prompts = np.log(softmax(100 * e @ T.T) + 1e-9)
    p = softmax((1 - ALPHA[20]) * log_probe + ALPHA[20] * log_prompts)[0]
    top3 = [(BREEDS[i], round(float(p[i]), 2)) for i in np.argsort(-p)[:3]]
    zone = "always check" if top3[0][0] in ALWAYS_CHECK else "check" if top3[0][1] < LINE else "auto-fill"
    return top3, zone

NEW_SHOWN = ["beagle_44", "Bengal_43", "Abyssinian_92", "British_Shorthair_84", "american_pit_bull_terrier_197", "chihuahua_60"]
show([f"images/{n}.jpg" for n in NEW_SHOWN], [n.rsplit("_", 1)[0] for n in NEW_SHOWN])
for n in NEW_SHOWN:
    top3, zone = suggest(f"images/{n}.jpg")
    print(f"{n:32} {zone:13} " + ", ".join(f"{b} {p:.2f}" for b, p in top3))

> **Check.** A beagle, a Bengal and an Abyssinian are filled in automatically. The British Shorthair is filled in as a Russian Blue: a mistake that the line lets through. The pit bull goes to a person (always), and the chihuahua too, because its top probability is low.

Now all 148 new photos:

In [ ]:
results = pd.DataFrame([{"breed": b, **dict(zip(("top3", "zone"), suggest(f)))} for f, b in zip(files["new"], y["new"])])
results["right"] = [t[0][0] == b for t, b in zip(results.top3, results.breed)]
print(results.zone.value_counts().to_dict())
auto = results[results.zone == "auto-fill"]
print("auto-filled and wrong:", int((~auto.right).sum()), "of", len(auto))
print("the true breed is among the three suggestions:", sum(b in [x[0] for x in t] for t, b in zip(results.top3, results.breed)), "of", len(results))

> **Check.** 120 filled in automatically (4 of them wrong), 16 checked because of the line, 12 always checked. The true breed is among the three suggestions for 146 of the 148 photos, so the person who checks usually picks from three.

## Next

The [lesson page](https://learning.nextia-ai.com/courses/vision/m09/classify-products-and-pets-from-photos/) has the decision, the limits, responsible use, how to make this a portfolio piece, and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Count and detect in crowded scenes](https://learning.nextia-ai.com/courses/vision/m09/count-and-detect-in-crowded-scenes/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/vision/m09/classify-products-and-pets-from-photos/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The photos are from the Oxford-IIIT Pet Dataset (Parkhi, Vedaldi, Zisserman and Jawahar), CC BY-SA 4.0; the copyright of each photo stays with its owner. Photos in which a person detector found a person are not used.